In [ ]:
import pandas as pd

df_maceio = pd.read_csv(
    "./output/maceio/filtered_maceio.csv", delimiter=";", low_memory=False
)

In [70]:
display(df_maceio)

,LONGITUDE,LATITUDE,NATUREZA_FATO,DATA_HORA,CIDADE_FATO
0,-35.698388,-9.658217,ROUBO A TRANSEUNTE,2022-01-01 02:00:00,Maceió
1,-35.703405,-9.660662,ROUBO A TRANSEUNTE,2022-01-01 02:00:00,Maceió
2,-35.710142,-9.653007,ROUBO DE VEÍCULO (MOTO),2022-01-01 02:00:00,Maceió
3,-35.713147,-9.668955,ROUBO A TRANSEUNTE,2022-01-01 03:00:00,Maceió
4,-35.703983,-9.663635,ROUBO A TRANSEUNTE,2022-01-01 04:00:00,Maceió
...,...,...,...,...,...
83607,-35.756315,-9.662037,ROUBO A TRANSEUNTE,2014-12-22 12:50:00,Maceió
83608,-35.715706,-9.652401,ROUBO DE VEÍCULO (DE PASSEIO),2014-12-22 19:00:00,Maceió
83609,-35.755908,-9.586733,ROUBO A RESIDÊNCIA,2014-12-23 05:30:00,Maceió
83610,-35.738317,-9.624963,ROUBO A TRANSEUNTE,2014-12-23 09:30:00,Maceió


In [ ]:
import osmnx as ox
import geopandas as gpd

place = "Maceió, Alagoas, Brazil"
print(f"Querying OSM for: {place}")

gdf_place = ox.geocode_to_gdf(place)
polys = gdf_place[gdf_place.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    # fallback: fetch administrative boundaries and pick candidates
    tags = {"boundary": "administrative"}
    adm = ox.geometries_from_place(place, tags=tags)
    polys = adm[adm.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    raise SystemExit(
        "Could not find polygon for Maceió on OSM. Check query or network."
    )

# # compute area in metric projection to pick the largest polygon
polys = polys.to_crs("EPSG:3857")
polys["area_m2"] = polys.geometry.area
largest_idx = polys["area_m2"].idxmax()
poly = polys.loc[largest_idx].geometry
# convert back to WGS84
poly = gpd.GeoSeries([poly], crs="EPSG:3857").to_crs("EPSG:4326").iloc[0]

osm_maceio = gpd.GeoDataFrame({"name": ["Maceió"], "geometry": [poly]}, crs="EPSG:4326")
osm_path = "./output/maceio/osm_maceio.geojson"
osm_maceio.to_file(osm_path, driver="GeoJSON")

Querying OSM for: Maceió, Alagoas, Brazil


In [ ]:
# Spatial join points with municipality polygons (geojs-27-mun.json)
# Requires: geopandas
import geopandas as gpd

# load municipalities and ensure CRS is EPSG:4326
mun = gpd.read_file("./output/maceio/osm_maceio.geojson")
mun = mun.to_crs("EPSG:4326")

# create GeoDataFrame for Maceió points
gdf_maceio = gpd.GeoDataFrame(
    df_maceio.copy(),
    geometry=gpd.points_from_xy(df_maceio["LONGITUDE"], df_maceio["LATITUDE"]),
    crs="EPSG:4326",
)

# spatial join: attach municipality 'name' to each point (left join)
# use predicate='within' so points must fall inside polygon
joined = gpd.sjoin(
    gdf_maceio, mun[["name", "geometry"]], how="left", predicate="within"
)

# boolean: specifically in Maceió municipality
joined["in_maceio"] = joined["name"] == "Maceió"

# summary counts
total = len(df_maceio)
inside_count = int(joined["in_maceio"].sum())
outside_count = total - inside_count
print(f"Maceió records total: {total}")
print(f" - inside Maceió polygon: {inside_count}")
print(f" - outside Maceió polygon: {outside_count}")

# Filter df_maceio to only records whose points lie inside the Maceió polygon
inside_maceio = joined.loc[joined["in_maceio"]].copy()

inside_maceio_clean = inside_maceio.drop(
    columns=["geometry", "name", "in_maceio", "index_right"]
)

# reset index and overwrite df_maceio with filtered rows
df_maceio = inside_maceio_clean.reset_index(drop=True)

# save filtered CSV
output_path = "./output/maceio/filtered_maceio_inside_polygon.csv"
df_maceio.to_csv(output_path, index=False, sep=";")
print(f"Saved {len(df_maceio)} Maceió rows inside polygon to {output_path}")

Maceió records total: 83612
 - inside Maceió polygon: 83512
 - outside Maceió polygon: 100
Saved 83512 Maceió rows inside polygon to ./output/filtered_maceio_inside_polygon.csv


In [ ]:
import folium

# Show the map with the polygon and the points inside/outside the polygon
osm = gpd.read_file("./output/maceio/osm_maceio.geojson").to_crs("EPSG:4326")
centroid = osm.geometry.centroid.iloc[0]
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11)

folium.GeoJson(
    osm.to_json(),
    name="Maceió (OSM)",
    style_function=lambda f: {
        "fillColor": "transparent",
        "color": "blue",
        "weight": 2,
        "opacity": 0.8,
    },
).add_to(m)

# optional: overlay sample points if gdf_maceio exists
if "gdf_maceio" in globals():
    for _, r in joined.iterrows():
        folium.CircleMarker(
            [r.geometry.y, r.geometry.x],
            radius=2,
            color="green" if r["in_maceio"] else "red",
            fill=True,
        ).add_to(m)
m.save("./output/maceio/maceio_and_points.html")
print("Saved maceio_osm_polygon_map.html")

/tmp/ipykernel_12970/572557471.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = osm.geometry.centroid.iloc[0]


Saved maceio_osm_polygon_map.html
